# QLoRA Fine-Tuning for Medical Purpose

**QLoRA = Quantized LoRA** — the most memory-efficient fine-tuning method available.  
It combines two techniques:
1. **4-bit Quantization** — compress the frozen base model from 16-bit → 4-bit (4x smaller)
2. **LoRA** — inject tiny trainable adapter matrices on top of the frozen quantized model

## The QLoRA Formula

$$\text{QLoRA} = \underbrace{\text{NF4-Quantized}(W)}_{\text{frozen, 4-bit}} + \underbrace{B \cdot A}_{\text{trainable LoRA, fp16}}$$

- **W** (base model weights) → quantized to **4-bit NormalFloat (NF4)** — never updated  
- **A, B** (LoRA matrices) → kept in **fp16** — these are trained  
- **NF4** = NormalFloat4, a data type designed specifically for normally-distributed neural network weights

## Memory Breakdown for a 7B Model

| Component | Precision | Size |
|---|---|---|
| Base model weights (QLoRA) | 4-bit NF4 | ~3.5 GB |
| LoRA adapter (A + B matrices) | fp16 | ~30 MB |
| Optimizer states (only for LoRA) | fp32 | ~120 MB |
| **Total** | | **~4 GB** |
| Full fine-tune (comparison) | fp16 | ~56 GB |

## Key Hyperparameters

| Parameter | What it does | Typical Values |
|---|---|---|
| `load_in_4bit` | Enable 4-bit quantization | True |
| `bnb_4bit_quant_type` | Quantization data type | `"nf4"` (best for LLMs) |
| `bnb_4bit_compute_dtype` | Compute precision during forward pass | `torch.float16` |
| `bnb_4bit_use_double_quant` | Quantize the quantization constants too | True (saves ~0.4 bits extra) |
| `r` (LoRA rank) | Size of low-rank matrices | 4, 8, 16, 64 |
| `lora_alpha` | Scaling factor | 16, 32 (= 2×r) |

## Step 1 — Install Dependencies

In [ ]:
!pip install transformers peft datasets bitsandbytes accelerate trl torch

## Step 2 — Prepare Medical Dataset

Same format as LoRA — instruction + input + structured output.  
**Remember**: Patient names come from the **prompt at runtime**, not from training data.  
QLoRA learns the **format, style, and domain rules** from these examples.

In [ ]:
import json
import os

medical_data = [
    {
        "instruction": "Summarize the patient diagnosis",
        "input": "Patient: Alice, 32F. BP: 120/80. Sugar: 95 mg/dL. Temp: 98.6F. Complaint: routine checkup.",
        "output": """Patient: Alice | Age: 32 | Gender: Female
Vitals: BP: 120/80 (Normal) | Sugar: 95 mg/dL (Normal) | Temp: 98.6F (Normal)
Diagnosis: Healthy — No abnormalities found
Medication: None required
Follow-up: Annual checkup in 12 months"""
    },
    {
        "instruction": "Summarize the patient diagnosis",
        "input": "Patient: Bob, 58M. BP: 162/100. Sugar: 310 mg/dL. HbA1c: 9.2%. Complaint: fatigue, frequent urination.",
        "output": """Patient: Bob | Age: 58 | Gender: Male
Vitals: BP: 162/100 (⚠️ HIGH) | Sugar: 310 mg/dL (⚠️ HIGH) | HbA1c: 9.2% (⚠️ HIGH)
Diagnosis: Stage 2 Hypertension + Type 2 Diabetes
Medication: Amlodipine 5mg (once daily) | Metformin 500mg (twice daily)
Follow-up: 2 weeks — recheck BP and sugar levels"""
    },
    {
        "instruction": "Suggest medication",
        "input": "Patient has Type 2 Diabetes. HbA1c: 8.5%. No kidney issues. No allergies.",
        "output": """Recommended Medication Protocol:
First-line: Metformin 500mg twice daily with meals
Target HbA1c: < 7.0%
Monitoring: Check HbA1c every 3 months
Lifestyle: Low-carb diet, 30 min exercise daily
Warning signs: Hypoglycemia (sugar < 70) — reduce dose"""
    },
    {
        "instruction": "Identify risk factors",
        "input": "Patient: 65M, smoker, BMI 32, BP 145/92, family history of heart disease, sedentary lifestyle.",
        "output": """Risk Assessment:
🔴 HIGH RISK — Cardiovascular Event
Risk Factors Identified:
1. Age > 60 (non-modifiable)
2. Smoking (modifiable) — immediate cessation advised
3. Obesity — BMI 32 (modifiable) — target BMI < 25
4. Hypertension — BP 145/92 (modifiable) — medication required
5. Family history (non-modifiable)
6. Sedentary lifestyle (modifiable)
10-Year CVD Risk Score: ~28%
Action: Cardiology referral within 1 week"""
    },
    {
        "instruction": "Summarize the patient diagnosis",
        "input": "Patient: Mary, 45F. BP: 135/85. Cholesterol: 240 mg/dL. LDL: 160. Complaint: headache, dizziness.",
        "output": """Patient: Mary | Age: 45 | Gender: Female
Vitals: BP: 135/85 (⚠️ ELEVATED) | Cholesterol: 240 mg/dL (⚠️ HIGH) | LDL: 160 (⚠️ HIGH)
Diagnosis: Pre-hypertension + Hypercholesterolemia
Medication: Atorvastatin 10mg (once daily at night)
Lifestyle: DASH diet, reduce saturated fats, 30 min walking daily
Follow-up: 4 weeks — recheck lipid panel"""
    }
]

def format_prompt(item):
    return f"""### Instruction:
{item['instruction']}

### Input:
{item['input']}

### Response:
{item['output']}"""

formatted = [{"text": format_prompt(item)} for item in medical_data]

os.makedirs("data", exist_ok=True)
with open("data/medical_qlora.json", "w") as f:
    json.dump(formatted, f, indent=2)

print(f"✅ Dataset created: {len(formatted)} samples")
print("\nSample entry:\n")
print(formatted[1]["text"])

## Step 3 — Configure 4-bit Quantization (The QLoRA-Specific Part)

This is the **key difference** from plain LoRA.  
We use `BitsAndBytesConfig` to load the base model in **4-bit NF4** format.

### How NF4 Quantization Works
```
Original weight:  0.3842 (16-bit float, 2 bytes)
After NF4:        1011   (4-bit integer, 0.5 bytes)
                   ↑ 4x compression!

During forward pass: dequantize back to fp16 for computation
During backward pass: only LoRA A,B gradients are computed (base model is frozen)
```

### Double Quantization (extra saving)
```
Normal:  quantize weights          → saves ~8 GB on 70B model
Double:  also quantize the scale   → saves extra ~0.37 GB
```

In [ ]:
import torch
from transformers import BitsAndBytesConfig

# ─── QLoRA-SPECIFIC: 4-bit quantization config ────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                        # ← THE KEY QLORA FLAG (LoRA doesn't have this)
    bnb_4bit_quant_type="nf4",                # NormalFloat4 — optimal for neural network weights
    bnb_4bit_compute_dtype=torch.float16,     # dequantize to fp16 during forward pass
    bnb_4bit_use_double_quant=True            # quantize scale constants too (saves ~0.4 bits more)
)

print("QLoRA 4-bit config created:")
print(f"  Quant type:      {bnb_config.bnb_4bit_quant_type}")
print(f"  Compute dtype:   {bnb_config.bnb_4bit_compute_dtype}")
print(f"  Double quant:    {bnb_config.bnb_4bit_use_double_quant}")
print()
print("Memory impact on LLaMA 3 8B:")
print("  Without quant:  ~16 GB  (fp16)")
print("  With NF4:       ~4 GB   (4-bit)  ← 4x reduction!")

## Step 4 — Load Base Model with QLoRA Quantization

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import prepare_model_for_kbit_training

# ─── CONFIG ───────────────────────────────────────────────
BASE_MODEL  = "meta-llama/Llama-3-8B-Instruct"  # or "mistralai/Mistral-7B-Instruct-v0.3"
OUTPUT_DIR  = "./medical-qlora-adapter"
DATA_PATH   = "./data/medical_qlora.json"
MAX_SEQ_LEN = 512
# ──────────────────────────────────────────────────────────

print("Loading base model in 4-bit NF4 (QLoRA)...")

# Pass bnb_config here — this is what makes it QLoRA instead of LoRA
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,   # ← 4-bit quantization applied here
    device_map="auto",                # distributes across GPU/CPU automatically
    trust_remote_code=True
)

# Critical step for QLoRA — enables gradient checkpointing on quantized model
model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("✅ Base model loaded in 4-bit (QLoRA mode)")
print(f"Model dtype: {next(model.parameters()).dtype}")  # should show torch.uint8 or torch.float16

## Step 5 — Attach LoRA Adapters on top of Quantized Model

Same LoRA config as before — but now it sits on top of the **4-bit quantized** base model.  
The adapters themselves are in **fp16** for training precision.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=64,                         # higher rank = more capacity (can afford with QLoRA memory savings)
    lora_alpha=16,                # scaling = alpha/r
    target_modules=[              # attention layers to inject LoRA into
        "q_proj",                 # Query
        "k_proj",                 # Key
        "v_proj",                 # Value
        "o_proj",                 # Output
        "gate_proj",              # FFN gate  — extra layers for better quality
        "up_proj",                # FFN up
        "down_proj",              # FFN down
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
# Expected:
# trainable params: ~41,943,040 || all params: ~8,072,204,288 || trainable%: ~0.52%

## Step 6 — Train with QLoRA

`paged_adamw_32bit` is the recommended optimizer for QLoRA — it pages optimizer states to CPU RAM when GPU memory is tight.

In [ ]:
import json
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer

# Load dataset
with open(DATA_PATH) as f:
    data = json.load(f)
dataset = Dataset.from_list(data)
print(f"Training samples: {len(dataset)}")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,       # effective batch = 8
    warmup_ratio=0.03,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    save_strategy="epoch",
    optim="paged_adamw_32bit",           # ← QLoRA-specific: pages optimizer states to CPU RAM
    lr_scheduler_type="cosine",          # cosine decay for smoother convergence
    report_to="none",
    group_by_length=True                 # group similar-length samples → less padding → faster
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=False
)

print("Starting QLoRA training...")
trainer.train()
print("✅ QLoRA Training complete!")

## Step 7 — Save the QLoRA Adapter

In [ ]:
# Save only the LoRA adapter (NOT the quantized base model)
# The adapter is fp16 — ~30-100MB depending on rank
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

import os
print(f"✅ QLoRA adapter saved to: {OUTPUT_DIR}")
print(f"Files: {os.listdir(OUTPUT_DIR)}")
# adapter_model.safetensors  ← tiny adapter file
# adapter_config.json        ← stores r, lora_alpha, target_modules etc.

## Step 8 — Inference with QLoRA Fine-Tuned Model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL   = "meta-llama/Llama-3-8B-Instruct"
ADAPTER_PATH = "./medical-qlora-adapter"

# For inference, reload the model in 4-bit again (memory efficient)
bnb_inf = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_inf,
    device_map="auto"
)

# Attach QLoRA adapter
model_inf = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model_inf.eval()

tokenizer_inf = AutoTokenizer.from_pretrained(ADAPTER_PATH)

def diagnose(instruction: str, patient_data: str) -> str:
    prompt = f"""### Instruction:
{instruction}

### Input:
{patient_data}

### Response:
"""
    inputs = tokenizer_inf(prompt, return_tensors="pt").to(model_inf.device)

    with torch.no_grad():
        outputs = model_inf.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer_inf.eos_token_id
        )

    response = tokenizer_inf.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return response.strip()

print("✅ QLoRA model ready for inference")

## Step 9 — Test with New Patients

In [ ]:
# Test 1 — New patient diagnosis
result1 = diagnose(
    instruction="Summarize the patient diagnosis",
    patient_data="Patient: John, 45M. BP: 155/95. Sugar: 290 mg/dL. HbA1c: 8.8%. Complaint: blurred vision, fatigue."
)
print("=" * 60)
print("TEST 1 — DIAGNOSIS:")
print(result1)

# Test 2 — Medication suggestion
result2 = diagnose(
    instruction="Suggest medication",
    patient_data="Patient has Stage 1 Hypertension. BP: 138/88. Age 50, no diabetes, no kidney issues."
)
print("\n" + "=" * 60)
print("TEST 2 — MEDICATION:")
print(result2)

# Test 3 — Risk assessment
result3 = diagnose(
    instruction="Identify risk factors",
    patient_data="Patient: Sarah, 55F. Smoker. BMI 29. BP 148/94. Mother had stroke at 60."
)
print("\n" + "=" * 60)
print("TEST 3 — RISK FACTORS:")
print(result3)

# LoRA vs QLoRA — Complete Difference

---

## 1. Core Concept

| | LoRA | QLoRA |
|---|---|---|
| **Full form** | Low-Rank Adaptation | Quantized Low-Rank Adaptation |
| **Invented by** | Hu et al., 2021 (Microsoft) | Dettmers et al., 2023 (UW) |
| **Base model weights** | Frozen in **fp16** (16-bit) | Frozen in **4-bit NF4** |
| **Adapter weights** | Trained in fp16 | Trained in fp16 |
| **Extra step** | None | 4-bit quantization of base model |

---

## 2. Memory Usage (LLaMA 3 8B)

| Component | LoRA | QLoRA |
|---|---|---|
| Base model | ~16 GB (fp16) | ~4 GB (4-bit NF4) |
| LoRA adapter | ~30 MB | ~30 MB |
| Optimizer states | ~500 MB | ~120 MB (paged to CPU) |
| Activations | ~2 GB | ~1 GB |
| **Total** | **~18–20 GB** | **~5–6 GB** |

**QLoRA fits a 7-8B model on a 6GB GPU. LoRA needs 18–20GB.**

---

## 3. Code Difference — Side by Side

```python
# ─── LoRA ────────────────────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3-8B",
    torch_dtype=torch.float16,   # full fp16 — 16 GB
    device_map="auto"
)

# ─── QLoRA ───────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,           # ← THIS is the only real difference
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3-8B",
    quantization_config=bnb_config,  # 4-bit NF4 — 4 GB
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)  # ← extra QLoRA step
```

**After loading the model, everything else (LoraConfig, training, saving) is identical.**

---

## 4. Speed & Quality

| | LoRA | QLoRA |
|---|---|---|
| **Training speed** | Faster | ~20–30% slower (dequant overhead) |
| **Output quality** | Baseline | ~1–3% lower (minor quantization loss) |
| **Inference speed** | Normal | Slightly slower (dequant at runtime) |
| **Merge adapter** | ✅ Yes | ⚠️ Partial (dequantize first) |

---

## 5. When to Use Which

| Scenario | Use |
|---|---|
| GPU has ≥ 20GB VRAM (A100, RTX 4090) | **LoRA** — faster, slightly better quality |
| GPU has 6–12GB VRAM (RTX 3060–3090) | **QLoRA** — only way to fit the model |
| Cloud training (pay per GPU-hour) | **QLoRA** — fewer GPUs needed = cheaper |
| Production inference speed matters | **LoRA** — no dequantization overhead |
| Learning / experimenting at home | **QLoRA** — works on consumer GPU |
| 30B–70B model fine-tuning | **QLoRA** — only practical option |

---

## 6. Simple Analogy

```
LoRA   = Store a full-size blueprint (big), then sketch modifications on a sticky note
QLoRA  = Compress the blueprint to thumbnail (4x smaller), then sketch modifications on a sticky note

Result is almost the same. QLoRA just uses far less space to store the blueprint.
```

---

## Summary

> **QLoRA = LoRA + 4-bit quantization of the base model**  
> The LoRA adapter part is identical in both.  
> The only difference is HOW the frozen base model is stored in memory.  
> QLoRA trades a tiny bit of quality for a massive reduction in memory requirements.